# Working with the INbreast COCO Dataset

This notebook demonstrates how to load the INbreast mammography dataset using COCO-format annotations with the `medical_image` framework.

In [ ]:
!pip install medical-image-std

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

from medical_image import DicomImage

## 1. Dataset Structure

Our INbreast dataset uses a simple COCO-like layout:

```
data/Inbreast/
├── annotations.json     # COCO-format (images, annotations, categories)
└── images/
    ├── 20586934.dcm
    ├── 53582683.dcm
    └── ...
```

The annotations contain polygon segmentations for **microcalcifications**.

In [ ]:
# Set your dataset path
DATASET_ROOT = Path("data/Inbreast")  # adjust if needed

# Load the COCO annotations
with open(DATASET_ROOT / "annotations.json") as f:
    coco = json.load(f)

print(f"Images: {len(coco['images'])}")
print(f"Annotations: {len(coco['annotations'])}")
print(f"Categories: {coco['categories']}")
print(f"\nSample image entry: {coco['images'][0]}")

## 2. COCO Mammography Dataset Class

A PyTorch Dataset that loads DICOM images and renders polygon annotations as binary masks.

In [ ]:
class CocoMammographyDataset(Dataset):
    """PyTorch Dataset for COCO-annotated DICOM mammograms."""

    def __init__(self, root_dir, target_size=(512, 512), transform=None):
        self.root = Path(root_dir)
        self.target_size = target_size
        self.transform = transform

        with open(self.root / "annotations.json") as f:
            self.coco = json.load(f)

        self.images = self.coco["images"]
        # Group annotations by image_id
        self.img_to_anns = {}
        for ann in self.coco["annotations"]:
            self.img_to_anns.setdefault(ann["image_id"], []).append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info["id"]
        h, w = img_info["height"], img_info["width"]

        # Load DICOM
        dcm_path = self.root / "images" / img_info["file_name"]
        dcm = DicomImage(str(dcm_path))
        dcm.load()
        image = dcm.pixel_data.float().unsqueeze(0)  # (1, H, W)

        # Render polygon annotations as binary mask
        mask = self._render_mask(img_id, h, w)

        # Resize
        if self.target_size:
            image = F.interpolate(
                image.unsqueeze(0), size=self.target_size, mode="bilinear",
                align_corners=False
            ).squeeze(0)
            mask = F.interpolate(
                mask.unsqueeze(0), size=self.target_size, mode="nearest"
            ).squeeze(0)

        # Normalize image to [0, 1]
        img_min, img_max = image.min(), image.max()
        if img_max > img_min:
            image = (image - img_min) / (img_max - img_min)

        if self.transform:
            image = self.transform(image)

        return {
            "image": image,
            "mask": mask,
            "metadata": {
                "case_id": img_info["file_name"].replace(".dcm", ""),
                "file_name": img_info["file_name"],
                "num_annotations": len(self.img_to_anns.get(img_id, [])),
            },
        }

    def _render_mask(self, img_id, h, w):
        """Render all polygon annotations for an image as a binary mask."""
        mask = np.zeros((h, w), dtype=np.uint8)
        anns = self.img_to_anns.get(img_id, [])

        for ann in anns:
            for seg in ann.get("segmentation", []):
                if len(seg) < 6:
                    continue
                pts = np.array(seg).reshape(-1, 2).astype(np.int32)
                # Fill polygon using OpenCV-free scanline approach
                from skimage.draw import polygon as draw_polygon
                rr, cc = draw_polygon(pts[:, 1], pts[:, 0], shape=(h, w))
                mask[rr, cc] = 1

        return torch.from_numpy(mask).float().unsqueeze(0)  # (1, H, W)

## 3. Load and Inspect the Dataset

In [ ]:
dataset = CocoMammographyDataset(
    root_dir=DATASET_ROOT,
    target_size=(512, 512),
)

print(f"Dataset size: {len(dataset)} samples")

In [ ]:
# Access a single sample
sample = dataset[0]

image = sample["image"]
mask = sample["mask"]
meta = sample["metadata"]

print(f"Image shape: {image.shape}")
print(f"Mask shape:  {mask.shape}")
print(f"Image range: [{image.min():.3f}, {image.max():.3f}]")
print(f"Metadata:    {meta}")
print(f"Mask has lesions: {mask.sum().item() > 0}")

In [ ]:
# Visualize image and mask
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image.squeeze(0).numpy(), cmap="gray")
axes[0].set_title(f"Mammogram ({meta['case_id']})")
axes[1].imshow(mask.squeeze(0).numpy(), cmap="hot")
axes[1].set_title(f"MC Mask ({meta['num_annotations']} annotations)")
# Overlay
axes[2].imshow(image.squeeze(0).numpy(), cmap="gray")
axes[2].imshow(mask.squeeze(0).numpy(), cmap="Reds", alpha=0.4)
axes[2].set_title("Overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Using a DataLoader for Training

In [ ]:
loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)

batch = next(iter(loader))
print(f"Batch image shape: {batch['image'].shape}")  # (4, 1, 512, 512)
print(f"Batch mask shape:  {batch['mask'].shape}")   # (4, 1, 512, 512)

In [ ]:
# Visualize a batch
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0, i].imshow(batch["image"][i, 0].numpy(), cmap="gray")
    axes[0, i].set_title(f"Image {i}")
    axes[0, i].axis("off")
    axes[1, i].imshow(batch["mask"][i, 0].numpy(), cmap="hot")
    axes[1, i].set_title(f"Mask {i}")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## 5. Dataset Statistics

In [ ]:
# Annotation statistics
ann_counts = [len(dataset.img_to_anns.get(img["id"], [])) for img in dataset.images]

print(f"Total annotations: {sum(ann_counts)}")
print(f"Annotations per image: min={min(ann_counts)}, max={max(ann_counts)}, "
      f"mean={np.mean(ann_counts):.1f}")

plt.figure(figsize=(8, 3))
plt.hist(ann_counts, bins=20, edgecolor="black")
plt.xlabel("Annotations per image")
plt.ylabel("Count")
plt.title("Distribution of microcalcification annotations")
plt.tight_layout()
plt.show()

## 6. Using BaseDataset.from_coco_json()

The framework also provides a utility to parse COCO JSON files directly.

In [ ]:
from medical_image.datasets.base_dataset import BaseDataset

coco_data = BaseDataset.from_coco_json(str(DATASET_ROOT / "annotations.json"))

print(f"Images: {len(coco_data['images'])}")
print(f"Categories: {coco_data['categories']}")
print(f"Images with annotations: {len(coco_data['annotations'])}")

# Each annotation is reconstructed as an Annotation object
first_img_id = coco_data['images'][0]['id']
anns = coco_data['annotations'].get(first_img_id, [])
if anns:
    print(f"\nFirst image has {len(anns)} annotations")
    print(f"First annotation: {anns[0]}")